In [ ]:
# Data and Math Libraries
import os
os.environ.setdefault('POLARS_MAX_THREADS', '1')  # avoids a torch/polars native-thread deadlock on macOS
import polars as pl
import numpy as np
from datetime import datetime, timedelta
import random

#Machine Learning Libraries
import torch
import torch.nn as nn 
import torch.optim as optim
import research 
import models

#visualization 
import altair as alt

#data sources
import binance

In [3]:
print(research.__file__)
print([x for x in dir(research) if not x.startswith('_')])

/Users/arezziorietti/Documents/ML Projects/research.py
['np', 'random', 'set_seed', 'torch']


In [4]:
research.set_seed(42)

In [7]:
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_cols(-1) 

polars.config.Config

In [ ]:
sym = 'BTCUSDT'

hist_data_window = 7 * 4 * 6

time_interval = '1h'

max_lags = 4

forecast_horizon = 1 

annualized_rate = research.sharpe_annualization_factor(time_interval, 365, 24)

# Part 1: BTCUSDT Research

## Download & Load Price Data

In [ ]:
# Download hist_data_window days of trade data (skips days already cached)
binance.download_trades(sym, hist_data_window)

In [ ]:
ts = research.load_ohlc_timeseries(sym, time_interval)
ts

In [ ]:
research.plot_static_timeseries(ts, sym, 'close', time_interval)

## Feature Engineering

Build the target (future log return) and auto-regressive lag features.

In [ ]:
target = 'close_log_return'
ts = research.add_log_return_features(ts, 'close', forecast_horizon, max_no_lags=max_lags)
ts = ts.drop_nulls()
ts

In [ ]:
research.plot_distribution(ts, target, no_bins=100)

In [ ]:
# Auto-correlation between the target and its own lags
research.auto_reg_corr_matrx(ts, target, max_lags)

## Benchmark Linear Models

Try every combination of up to 3 lag features and rank by annualized Sharpe.

In [ ]:
feature_pool = [f'{target}_lag_{i}' for i in range(1, max_lags + 1)]
btc_benchmark = research.benchmark_linear_models(
    ts, target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss()
)
btc_benchmark

In [ ]:
btc_best = btc_benchmark.row(0, named=True)
btc_best_features = btc_best['features'].split(',')
btc_best_features

## Best Model: Trade-Level Performance

In [ ]:
btc_model = models.LinearModel(len(btc_best_features))
btc_model_trades = research.learn_model_trades(ts, btc_best_features, target, btc_model, loss=nn.L1Loss())
research.plot_column(btc_model_trades, 'equity_curve')

## Add Transaction Fees

In [ ]:
maker_fee = 0.00045
taker_fee = 0.00045
btc_model_trades = research.add_tx_fees_log(btc_model_trades, maker_fee, taker_fee)
research.plot_column(btc_model_trades, 'equity_curve_net_taker')

## Save Model

In [ ]:
torch.save(btc_model.state_dict(), 'BTCUSDT_model_weights.pth')
print(f"Saved BTCUSDT model with features {btc_best_features}")

## Out-of-Sample Validation (Walk-Forward)

The benchmark above picked the "best" feature combo by scanning every lag combination and choosing whichever had the highest Sharpe on a single holdout split. With that many combinations tried, a good-looking Sharpe can show up by luck alone (data snooping) — it doesn't prove the model has real predictive power.

This section fixes that: `walk_forward_eval` re-runs feature selection and training on rolling windows where each fold's test period was never seen during that fold's selection or training. The folds are then stitched into one continuous out-of-sample equity curve, which we compare against two baselines: buy-and-hold, and a random coin-flip trading signal (the null hypothesis of "no skill at all").

In [ ]:
btc_wf_folds, btc_oos_y_test, btc_oos_y_hat = research.walk_forward_eval(
    ts, target, feature_pool, annualized_rate, n_folds=5, loss=nn.L1Loss()
)
btc_wf_folds

In [ ]:
# Stitched-together out-of-sample performance across all folds
btc_oos_perf = research.eval_model_performance(btc_oos_y_test, btc_oos_y_hat, feature_pool, target, annualized_rate)
btc_oos_perf

### Baselines: Buy & Hold and a Random-Signal Null Distribution

In [ ]:
btc_bh_perf = research.buy_and_hold_performance(btc_oos_y_test, annualized_rate)
btc_bh_perf

In [ ]:
# Null distribution: 1000 random +1/-1 signals against the SAME realized OOS returns.
# If our model has no real edge, its Sharpe should look like a typical draw from this distribution.
btc_null_sharpes = research.random_signal_sharpes(btc_oos_y_test, annualized_rate, n_trials=1000, seed=42)
btc_p_value = research.sharpe_p_value(btc_oos_perf['sharpe'], btc_null_sharpes)

print(f"BTCUSDT OOS model Sharpe:      {btc_oos_perf['sharpe']:.2f}")
print(f"BTCUSDT Buy & Hold Sharpe:     {btc_bh_perf['sharpe']:.2f}")
print(f"Random-signal null Sharpe:     mean={btc_null_sharpes.mean():.2f}, std={btc_null_sharpes.std():.2f}")
print(f"P(random signal Sharpe >= model OOS Sharpe): {btc_p_value:.3f}")

In [ ]:
import matplotlib.pyplot as plt

btc_oos_trades = research.model_trade_results(btc_oos_y_test, btc_oos_y_hat)
btc_bh_equity_curve = np.cumsum(btc_oos_y_test.numpy().flatten())

plt.figure(figsize=(12, 6))
plt.plot(btc_oos_trades['equity_curve'], label='Model (walk-forward OOS)')
plt.plot(btc_bh_equity_curve, label='Buy & Hold')
plt.title('BTCUSDT: Out-of-Sample Equity Curve vs Buy & Hold')
plt.xlabel('Period (stitched across folds)')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()

# Part 2: ETHUSDT Research

Repeat the exact same research pipeline on a second asset (ETHUSDT) so we're not overfitting our conclusions to a single symbol.

## Download & Load Price Data

In [ ]:
sym = 'ETHUSDT'
binance.download_trades(sym, hist_data_window)

In [ ]:
eth_ts = research.load_ohlc_timeseries(sym, time_interval)
eth_ts

In [ ]:
research.plot_static_timeseries(eth_ts, sym, 'close', time_interval)

## Feature Engineering

In [ ]:
eth_ts = research.add_log_return_features(eth_ts, 'close', forecast_horizon, max_no_lags=max_lags)
eth_ts = eth_ts.drop_nulls()
eth_ts

In [ ]:
research.plot_distribution(eth_ts, target, no_bins=100)

In [ ]:
research.auto_reg_corr_matrx(eth_ts, target, max_lags)

## Benchmark Linear Models

In [ ]:
eth_benchmark = research.benchmark_linear_models(
    eth_ts, target, feature_pool, annualized_rate, max_no_features=3, loss=nn.L1Loss()
)
eth_benchmark

In [ ]:
eth_best = eth_benchmark.row(0, named=True)
eth_best_features = eth_best['features'].split(',')
eth_best_features

## Best Model: Trade-Level Performance

In [ ]:
eth_model = models.LinearModel(len(eth_best_features))
eth_model_trades = research.learn_model_trades(eth_ts, eth_best_features, target, eth_model, loss=nn.L1Loss())
research.plot_column(eth_model_trades, 'equity_curve')

## Add Transaction Fees

In [ ]:
eth_model_trades = research.add_tx_fees_log(eth_model_trades, maker_fee, taker_fee)
research.plot_column(eth_model_trades, 'equity_curve_net_taker')

## Save Model

In [ ]:
torch.save(eth_model.state_dict(), 'ETHUSDT_model_weights.pth')
print(f"Saved ETHUSDT model with features {eth_best_features}")

## Out-of-Sample Validation (Walk-Forward)

Same walk-forward validation as BTCUSDT: pick features and train fresh on each fold's training window only, evaluate once on that fold's unseen test window, then compare the stitched OOS result against buy-and-hold and a random-signal null distribution.

In [ ]:
eth_wf_folds, eth_oos_y_test, eth_oos_y_hat = research.walk_forward_eval(
    eth_ts, target, feature_pool, annualized_rate, n_folds=5, loss=nn.L1Loss()
)
eth_wf_folds

In [ ]:
eth_oos_perf = research.eval_model_performance(eth_oos_y_test, eth_oos_y_hat, feature_pool, target, annualized_rate)
eth_oos_perf

### Baselines: Buy & Hold and a Random-Signal Null Distribution

In [ ]:
eth_bh_perf = research.buy_and_hold_performance(eth_oos_y_test, annualized_rate)
eth_bh_perf

In [ ]:
eth_null_sharpes = research.random_signal_sharpes(eth_oos_y_test, annualized_rate, n_trials=1000, seed=42)
eth_p_value = research.sharpe_p_value(eth_oos_perf['sharpe'], eth_null_sharpes)

print(f"ETHUSDT OOS model Sharpe:      {eth_oos_perf['sharpe']:.2f}")
print(f"ETHUSDT Buy & Hold Sharpe:     {eth_bh_perf['sharpe']:.2f}")
print(f"Random-signal null Sharpe:     mean={eth_null_sharpes.mean():.2f}, std={eth_null_sharpes.std():.2f}")
print(f"P(random signal Sharpe >= model OOS Sharpe): {eth_p_value:.3f}")

In [ ]:
eth_oos_trades = research.model_trade_results(eth_oos_y_test, eth_oos_y_hat)
eth_bh_equity_curve = np.cumsum(eth_oos_y_test.numpy().flatten())

plt.figure(figsize=(12, 6))
plt.plot(eth_oos_trades['equity_curve'], label='Model (walk-forward OOS)')
plt.plot(eth_bh_equity_curve, label='Buy & Hold')
plt.title('ETHUSDT: Out-of-Sample Equity Curve vs Buy & Hold')
plt.xlabel('Period (stitched across folds)')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()

# Part 3: BTCUSDT vs ETHUSDT Comparison

In [ ]:
comparison = pl.concat([
    btc_benchmark.head(1).with_columns(pl.lit('BTCUSDT').alias('sym')),
    eth_benchmark.head(1).with_columns(pl.lit('ETHUSDT').alias('sym')),
]).select(['sym'] + [c for c in btc_benchmark.columns if c != 'sym'])
comparison

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(btc_model_trades['equity_curve_net_taker'], label='BTCUSDT')
plt.plot(eth_model_trades['equity_curve_net_taker'], label='ETHUSDT')
plt.title('Net (after taker fees) Equity Curve: BTCUSDT vs ETHUSDT')
plt.xlabel('Trade #')
plt.ylabel('Cumulative Log Return')
plt.legend()
plt.tight_layout()
plt.show()

# Part 4: Statistical Verdict — Is Any of This Actually Useful?

The equity curves and Sharpe ratios earlier in this notebook (the "Benchmark Linear Models" sections) were picked by scanning many feature combinations and keeping the best one — that number alone doesn't prove skill, since the best of many random tries tends to look good even with zero real edge.

This section pulls together the walk-forward *out-of-sample* results for both symbols and checks each one against two bars it has to clear to be called "useful":
1. Beat buy-and-hold (otherwise just holding the asset was better than trading it).
2. Beat the random-signal null distribution by a wide enough margin that the p-value is small (otherwise a coin flip could have produced the same result).

In [ ]:
verdict = pl.DataFrame([
    {
        'sym': 'BTCUSDT',
        'oos_sharpe': btc_oos_perf['sharpe'],
        'buy_and_hold_sharpe': btc_bh_perf['sharpe'],
        'beats_buy_and_hold': btc_oos_perf['sharpe'] > btc_bh_perf['sharpe'],
        'random_null_mean_sharpe': btc_null_sharpes.mean(),
        'p_value': btc_p_value,
        'statistically_significant_p<0.05': btc_p_value < 0.05,
    },
    {
        'sym': 'ETHUSDT',
        'oos_sharpe': eth_oos_perf['sharpe'],
        'buy_and_hold_sharpe': eth_bh_perf['sharpe'],
        'beats_buy_and_hold': eth_oos_perf['sharpe'] > eth_bh_perf['sharpe'],
        'random_null_mean_sharpe': eth_null_sharpes.mean(),
        'p_value': eth_p_value,
        'statistically_significant_p<0.05': eth_p_value < 0.05,
    },
])
verdict

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sym_label, null_sharpes, actual_sharpe, p in [
    (axes[0], 'BTCUSDT', btc_null_sharpes, btc_oos_perf['sharpe'], btc_p_value),
    (axes[1], 'ETHUSDT', eth_null_sharpes, eth_oos_perf['sharpe'], eth_p_value),
]:
    ax.hist(null_sharpes, bins=40, alpha=0.7, label='Random-signal null distribution')
    ax.axvline(actual_sharpe, color='red', linewidth=2, label=f'Model OOS Sharpe ({actual_sharpe:.2f})')
    ax.set_title(f'{sym_label}: Model Sharpe vs Chance (p={p:.3f})')
    ax.set_xlabel('Sharpe Ratio')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.show()